In [1]:
import tensorflow as tf
import numpy as np
import os
import tempfile
import zipfile

In [2]:
model_path = "models/base_models/cassava_edge_baseline_model.h5"

model = tf.keras.models.load_model(model_path)
print("Model loaded successfully.")

Model loaded successfully.


In [6]:
X_train = np.load("X_train.npy")

def representative_data_gen():
    for i in range(100):
        yield [X_train[i:i+1].astype(np.float32)]

In [7]:
MODEL_TFLITE_PATH = "models/optimized_models/cassava_edge_model_dynamic.tflite"

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# IMPORTANT: do NOT force full INT8
# (this is what makes it hybrid)

tflite_model_hybrid = converter.convert()

with open(MODEL_TFLITE_PATH, "wb") as f:
    f.write(tflite_model_hybrid)

print("Hybrid INT8 quantized model saved successfully.")

INFO:tensorflow:Assets written to: C:\Users\nickh\AppData\Local\Temp\tmp0cve5xi4\assets


INFO:tensorflow:Assets written to: C:\Users\nickh\AppData\Local\Temp\tmp0cve5xi4\assets


Saved artifact at 'C:\Users\nickh\AppData\Local\Temp\tmp0cve5xi4'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 64, 64, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  1220109328048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220109452736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114013344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114032768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114109712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114109536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114234352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114234000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114307728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1220114307552: TensorSpec(shape=(), dtype=tf.resource, name=None)


C:\Users\nickh\anaconda3\envs\ml_lab\lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Hybrid INT8 quantized model saved successfully.


In [8]:
def get_gzipped_model_size(file):
    temp = tempfile.NamedTemporaryFile(suffix=".zip", delete=False)
    temp.close()

    with zipfile.ZipFile(temp.name, "w", compression=zipfile.ZIP_DEFLATED) as f:
        f.write(file)

    size = os.path.getsize(temp.name)
    os.remove(temp.name)
    return size

fp_size = get_gzipped_model_size(model_path)
quant_size = get_gzipped_model_size(MODEL_TFLITE_PATH)

print("Size of Full Precision Model: {} Bytes".format(fp_size))
print("Size of Quantized Model: {} Bytes".format(quant_size))
print("Size reduction factor: {:.2f}x".format(fp_size / quant_size))

Size of Full Precision Model: 94531 Bytes
Size of Quantized Model: 10723 Bytes
Size reduction factor: 8.82x


In [9]:
# Load test data
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# Load TFLite model
tflite_interpreter = tf.lite.Interpreter(model_path=MODEL_TFLITE_PATH)
tflite_interpreter.allocate_tensors()

input_details = tflite_interpreter.get_input_details()
output_details = tflite_interpreter.get_output_details()

predictions = np.zeros(len(X_test), dtype=int)

for i in range(len(X_test)):
    val_batch = X_test[i]

    # Hybrid still uses float input
    val_batch = np.expand_dims(val_batch, axis=0).astype(input_details[0]["dtype"])

    tflite_interpreter.set_tensor(input_details[0]['index'], val_batch)
    tflite_interpreter.invoke()

    output = tflite_interpreter.get_tensor(output_details[0]['index'])
    predictions[i] = output.argmax()

correct = 0
for i in range(len(predictions)):
    if predictions[i] == y_test[i]:
        correct += 1

accuracy_score = correct / len(predictions)

# Load FP model
full_precision_model = tf.keras.models.load_model(model_path, compile=False)
full_precision_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

score = full_precision_model.evaluate(X_test, y_test, verbose=0)

print("Accuracy of quantized to int8 model is {}%".format(accuracy_score * 100))
print("Compared to float32 accuracy of {}%".format(score[1] * 100))
print("We have a change of {}%".format((accuracy_score - score[1]) * 100))

C:\Users\nickh\anaconda3\envs\ml_lab\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Accuracy of quantized to int8 model is 65.32710280373831%
Compared to float32 accuracy of 65.21028280258179%
We have a change of 0.11682000115652569%
